**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher

# Example Chapter 29: Option Pricing with the Binomial Model
In this example we price European calls and puts with the binomial option pricing model. At each node of the tree, we build a replicating portfolio of $\Delta$ shares of stock and \$B in risk-free bonds that has the same payoff as the option one step later. By no arbitrage, the option is worth the cost of that portfolio. With many steps, the binomial price converges to the Black-Scholes price, which we compute with `fm.black_scholes()`.

### Imports and Setup

In [ ]:
# Import packages
%pip install -q --upgrade farms

import farms as fm
import numpy as np

### Define the Binomial Tree Function
`farms` does not include a binomial tree, so we define a short function, `binomial_tree()`, below.

`binomial_tree()`

**Required Inputs**

- `option_type` — Option type, `"call"` or `"put"`. Example: `option_type='call'`.
- `S` — Current stock price. Example: `S=100`.
- `K` — Strike price. Example: `K=105`.
- `T` — Time to expiration in years. Example: `T=0.5`.
- `rf` — Effective annual risk-free rate $R_f$, as a decimal. Example: `rf=0.01`.
- `sigma` — Annual volatility of the stock return, as a decimal. Example: `sigma=0.25`.
- `n` — Number of time steps in the tree. Example: `n=3`.

**Outputs**

- `float` — The no-arbitrage price of the option.

**How it works**

1. Each step has length $\Delta t = T/n$. Over one step the stock moves up by the factor $u = e^{\sigma\sqrt{\Delta t}}$ or down by $d = 1/u$.
2. At expiration, the stock price after $k$ down moves is $S\,u^{n-k}d^{k}$, and the option is worth its payoff, $\max(S_T-K,0)$ for a call or $\max(K-S_T,0)$ for a put.
3. We then step backward through the tree. At each node, the replicating portfolio holds
$$\Delta = \frac{V_u - V_d}{S(u-d)}$$
shares, where $V_u$ and $V_d$ are the option values after an up and a down move, and
$$B = \frac{V_u - \Delta\, S u}{(1+R_f)^{\Delta t}}$$
dollars in bonds. The option value at the node is $\Delta S + B$.

NumPy works on every node at a given date at once, so the only loop is the one that steps backward through time.

In [ ]:
def binomial_tree(option_type, S, K, T, rf, sigma, n):
    dt = T / n                       # length of one step in years
    u = np.exp(sigma * np.sqrt(dt))  # up factor
    d = 1 / u                        # down factor

    # Stock prices at expiration, from all up moves (k = 0) to all down moves (k = n)
    k = np.arange(n + 1)
    stock = S * u ** (n - k) * d ** k

    # Option payoffs at expiration
    if option_type == 'call':
        value = np.maximum(stock - K, 0)
    elif option_type == 'put':
        value = np.maximum(K - stock, 0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    # Step backward one date at a time
    for i in range(n - 1, -1, -1):
        stock = stock[:i + 1] / u    # stock prices at date i
        value_up = value[:i + 1]     # option value after an up move
        value_down = value[1:i + 2]  # option value after a down move

        delta = (value_up - value_down) / (stock * (u - d))   # shares of stock
        B = (value_up - delta * stock * u) / (1 + rf) ** dt    # dollars in bonds
        value = delta * stock + B                              # cost of the replicating portfolio

    return float(value[0])

### Price a Call with a Three-Step Tree
We start with a small tree that you can check by hand: a stock with a current price of \$20, a strike price of \$21, three months to expiration, an effective annual risk-free rate of 1%, a volatility of 19.032%, and three steps.

In [ ]:
call_price = binomial_tree('call', S=20, K=21, T=3/12, rf=0.01, sigma=0.19032, n=3)
print(f'Call price (3 steps): {call_price:.4f}')

### Compare the Binomial Model with Black-Scholes
Now we use the inputs from the Chapter 29 Black-Scholes example: $S = 100$, $K = 105$, $T = 0.5$, $R_f = 1\%$, and $\sigma = 0.25$. With 50,000 steps the binomial prices match the Black-Scholes prices of \$5.1809 for the call and \$9.6598 for the put.

For full parameter details, see the [`farms` API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#black_scholes).

`black_scholes()`

**Required Inputs**

- `type` — Option type. Potential values are `"call"` and `"put"`. Example: `type='call'`.
- `S` — Current price of the underlying stock. Example: `S=100`.
- `K` — Strike price. Example: `K=105`.
- `T` — Time to expiration in years. Example: `T=0.5`.
- `rf` — Effective annual risk-free rate $R_f$, as a decimal. The function converts it to the continuously compounded rate $\ln(1+R_f)$. Example: `rf=0.01`.
- `sigma` — Annual volatility of the stock return, as a decimal. Example: `sigma=0.25`.

**Optional Inputs**

- None.

**Outputs**

- `float` — The Black-Scholes option price.

In [ ]:
S = 100        # current stock price
K = 105        # strike price
T = 0.5        # time to expiration in years (six months)
rf = 0.01      # effective annual risk-free rate (1%)
sigma = 0.25   # annual volatility (25%)
n = 50000      # number of steps

call_binomial = binomial_tree('call', S, K, T, rf, sigma, n)
put_binomial = binomial_tree('put', S, K, T, rf, sigma, n)

call_bs = fm.black_scholes('call', S, K, T, rf, sigma)
put_bs = fm.black_scholes('put', S, K, T, rf, sigma)

print(f'Call: binomial {call_binomial:.4f}, Black-Scholes {call_bs:.4f}')
print(f'Put:  binomial {put_binomial:.4f}, Black-Scholes {put_bs:.4f}')

### Try it
1. Price the Chapter 29 call with `n = 1`, `n = 3`, `n = 10`, and `n = 100`. How quickly does the binomial price approach the Black-Scholes price?
2. Price a put with the three-step inputs (`S=20`, `K=21`, `T=3/12`, `rf=0.01`, `sigma=0.19032`, `n=3`). Then check put-call parity: $C - P = S - K/(1+R_f)^T$.
3. Inside `binomial_tree()`, add `print(delta)` just before the `return` line and price the three-step call. What does the first $\Delta$ tell you about the replicating portfolio?
4. Raise `sigma` to `0.40`. How do the call and put prices change?